In [ ]:
# -*- coding: utf-8 -*-
#"""export_data_temperatura_cidades.ipynb

#Automatically generated by Colab.

#Original file is located at
#    <colab-notebook-url-removed>
#"""

import ee
import os
import sys
import copy
import math
import geemap.foliumap as geemap
# Trigger the authentication flow.
ee.Authenticate()
# Initialize the library.
ee.Initialize(project=os.environ.get('EE_PROJECT'))
#ee.Initialize(project=os.environ.get('EE_PROJECT'))
import geemap.core as geemap
# Cloud authentication.
from google.colab import auth
auth.authenticate_user()

In [ ]:
# =================== PARÂMETROS ===================
start = '1985-01-01'
end   = '2024-12-01'

asset_city    = 'projects/ee-rnvuefsppgm/assets/CAP_JOIN2'
allGeometries = ee.FeatureCollection(asset_city)

print("We loaded shp allGeometries with size >>>", allGeometries.size().getInfo())

# Pega propriedades CD_MUN e NM_MUN
cds = allGeometries.aggregate_array('CD_MUN').getInfo()
nms = allGeometries.aggregate_array('NM_MUN').getInfo()

print("Lista de municípios encontrados >>>")
for cod, nome in zip(cds, nms):
    print(f"{cod} - {nome}")

# -----------------------
# Parametrização de assets
# -----------------------
assetInput  = 'projects/geo-data-s/assets/RSEI_CITIES'
assetOutput = 'projects/geo-data-s/assets/RSEI_CITIES_PCA'
exportScale = 30
exportCRS   = None


In [ ]:
codMun = '2927408'
if codMun == '2927408':
    # Chama a função de exportação para cada código de município
    print(f" processing a cidade >> {codMun}")
    featCol = allGeometries.filter(ee.Filter.eq('CD_MUN', codMun))
    feat = featCol.first()
    mask_feat = featCol.reduceToImage(['id_cod'], ee.Reducer.first()).gt(0)
    city_geom = featCol.geometry()
    print("show properties ", city_geom.getInfo())

In [ ]:
ic_spectal = ee.ImageCollection(assetInput)
print(f'We loaded ic_spectal with size >>> # {ic_spectal.size().getInfo()}')
lst_bands = ic_spectal.first().bandNames().getInfo()
print("the band Names are \n >> ")
print(lst_bands)


In [ ]:
# **************************************************************************
# Function to calculate Principal Components
# Code adapted from https://developers.google.com/earth-engine/guides/arrays_eigen_analysis
# **************************************************************************
# // This function accepts mean centered imagery, a scale and
# // a region in which to perform the analysis.  It returns the
# // Principal Components (PC) in the region as a new image.
def getPrincipalComponents(centered, nregion , nNN):
    # // Collapse the bands of the image into a 1D array per pixel.
    arrays = centered.toArray();

    # This helper function returns a list of new band names.
    lstBnd_sd = [f'sd{kk}' for kk in range(1, nNN)]
    lstBnd_pc = [f'PC{kk}' for kk in range(1, nNN + 1)]

    # // Compute the covariance of the bands within the region.
    covar = arrays.reduceRegion(
        reducer= ee.Reducer.centeredCovariance(),
        geometry= nregion,
        scale= 30,
        maxPixels= 1e13
    );

    #  Get the 'array' covariance result and cast to an array.
    #  This represents the band-to-band covariance within the region.
    covarArray = ee.Array(covar.get('array'));

    #  Perform an eigen analysis and slice apart the values
    # and vectors.
    eigens = covarArray.eigen();

    # This is a P-length vector of Eigenvalues.
    eigenValues = eigens.slice(1, 0, 1);

    # This is a PxP matrix with eigenvectors in rows.
    eigenVectors = eigens.slice(1, 1);

    # Convert the array image to 2D arrays for matrix computations.
    arrayImage = arrays.toArray(1);

    # Left multiply the image array by the matrix of eigenvectors.
    principal_components = ee.Image(eigenVectors).matrixMultiply(arrayImage)

    # Turn the square roots of the Eigenvalues into a P-band image.
    sd_image = (
        ee.Image(eigenValues.sqrt())
        .arrayProject([0])
        .arrayFlatten([lstBnd_sd])
    )

    # Left multiply the image array by the matrix of eigenvectors.
    principalComponents = ee.Image(eigenVectors).matrixMultiply(arrayImage);

    # Turn the square roots of the Eigenvalues into a P-band image.
    # Call abs() to turn negative eigenvalues to positive before
    # taking the square root
    sdImage = (ee.Image(eigenValues.abs().sqrt())
                .arrayProject([0])
                .arrayFlatten([lstBnd_pc]))

    # Turn the PCs into a P-band image, normalized by SD.
    return (principalComponents
            #   // Throw out an an unneeded dimension, [[]] -> [].
            .arrayProject([0])
            #   // Make the one band array image a multi-band image, [] -> image.
            .arrayFlatten([lstBnd_pc])
            #   // Normalize the PCs by their SDs.
            .divide(sdImage)
            # .set(varianceDict)
        )



In [ ]:
def makePCABands(maskedImage, mregion):
    lst_bands_ind_spect = ['NDBSI', 'NDVI', 'LST', 'wetness']
    NN = len(lst_bands_ind_spect)
    image = maskedImage.unmask()
    scale = 30

    # bandNames = image.bandNames();
    ###############################################################
    # Mean center the data to enable a faster covariance reducer ##
    # and an SD stretch of the principal components.             ##
    ###############################################################
    meanDict = image.reduceRegion(
        reducer= ee.Reducer.mean(),
        geometry= ee.Geometry(mregion),
        scale= scale,
        maxPixels= 1e13,
    );
    means = ee.Image.constant(meanDict.values(lst_bands_ind_spect));
    centered = image.subtract(means);


    pcImage = getPrincipalComponents(centered, mregion, NN);
    return pcImage.mask(maskedImage.mask());

In [ ]:
lst_idcod = ic_spectal.reduceColumns(ee.Reducer.toList(), ['system:index']).get('list').getInfo()
print(f" exporting image to asset << {assetOutput} >> ")

for cc, id_cod in enumerate(lst_idcod):
    print(f" # {cc} procesing >> {id_cod}")
    if cc > 10:
        temp_img = ee.Image(ic_spectal.filter(ee.Filter.eq('system:index', id_cod)).first())
        myGeom = temp_img.geometry()
        temp_img = makePCABands(temp_img, myGeom)
        # temp_img = temp_img.set('system:index', id_cod)
        print(temp_img.bandNames().getInfo())


        temp_img = (temp_img
            .copyProperties(myGeom, ['NM_MUN', 'CD_MUN'])
            .set('system:footprint', myGeom)
        )
        idSystem = id_cod + "_pca"
        is_asset_exp = assetOutput + '/' + idSystem


        task = ee.batch.Export.image.toAsset(
                        image= ee.Image(temp_img),
                        description= idSystem,
                        assetId= is_asset_exp,
                        region= myGeom.getInfo()['coordinates'],
                        pyramidingPolicy= {".default": "mode"},
                        scale= 30,
                        maxPixels= 1e13
                    )
        task.start()
        print(f" exporting image << {idSystem} >> to drive")